# Single-omics analysis

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering, SpectralClustering
from sklearn.metrics import silhouette_score, adjusted_mutual_info_score
from sklearn.preprocessing import StandardScaler
import ast
import itertools
from scipy.stats import kruskal
from ptitprince import PtitPrince as pt
from scipy.stats import hmean
import matplotlib.lines as mlines

In [ ]:
from matplotlib import rcParams
sns.set_theme(style='ticks')
rcParams.update({
    'font.size': 11,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.8,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'legend.fontsize': 10,
    'legend.frameon': False,
    'savefig.format': 'svg',
    'savefig.dpi': 300,  # Still useful for rasterized elements
    'figure.dpi': 100,
    'figure.figsize': (3.5, 2.5),  # Approx. half-column width
    'figure.constrained_layout.use': True,
    'svg.fonttype': 'none',  # Keep text as editable text (not paths)
    'axes.spines.top': False,
    'axes.spines.right': False,
})
colorblind_palette = sns.color_palette('colorblind')

In [ ]:
# Get patient sampling used in experiments
sampling = pd.read_csv("../results/cluster_analysis/benchmarking_files/firstbench_2clusters.csv")
patients = list(sampling.iloc[:10, sampling.columns.get_loc("y_pred_idx")])

In [ ]:
# Collect preprocessed data (run data_preprocessing.py script for samples with all modalities). Name dfs (useful for later)
methyl_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_Methylation.csv", index_col=0)
methyl_data.name = "Methylation"
cna_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_CNA.csv", index_col=0)
cna_data.name = "CNA"
mirna_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_miRNA.csv", index_col=0)
mirna_data.name = "miRNA"
mutation_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_Mutation.csv", index_col=0)
mutation_data.name = "Mutation"
rnaseq_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_RNAseq.csv", index_col=0)
rnaseq_data.name = "RNAseq"
rppa_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_RPPA.csv", index_col=0)
rppa_data.name = "RPPA"

In [ ]:
# Parameters for clustering
modalities = [methyl_data, cna_data, mirna_data, mutation_data, rnaseq_data, rppa_data]
clusters = [2, 3, 4, 5]
algorithms = ["Hierarchical", "Spectral"]
RANDOM_STATE = 42

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Perform clustering
rows = []
for algorithm in algorithms:
    for modality in modalities:
        for cluster in clusters:
            run_counter = 0
            for subset in patients:
                subset_patients = ast.literal_eval(subset)
                subset_data = modality.loc[subset_patients]
                subset_array = subset_data.to_numpy()
                scaled_df = StandardScaler().fit_transform(subset_array)
                if algorithm == "Hierarchical":
                    labels = AgglomerativeClustering(n_clusters=cluster).fit_predict(scaled_df)
                    silhouette = silhouette_score(X=scaled_df, labels=labels, random_state=RANDOM_STATE)
                elif algorithm == "Spectral":
                    labels = SpectralClustering(n_clusters=cluster, assign_labels="cluster_qr", random_state=RANDOM_STATE).fit_predict(scaled_df)
                    silhouette = silhouette_score(X=scaled_df, labels=labels, random_state=RANDOM_STATE)
                rows.append({"modality":modality.name, "algorithm":algorithm, "n_clusters":cluster, "run_n":run_counter, "n_samples":len(subset_patients), "y_pred":labels, "y_pred_idx":subset_patients, "silhouette":silhouette})
                run_counter += 1

In [ ]:
singleomics_df = pd.DataFrame(rows)

In [ ]:
# Function to calculate AMI score (edited from general_functions.py)
def calc_ami(df):
    df["sorted_y_pred_idx"] = df["y_pred_idx"].apply(sorted)
    df["sorted_y_pred"] = df.apply(lambda row: [row["y_pred"][row["y_pred_idx"].index(patient_id)] for patient_id in row["sorted_y_pred_idx"]], axis=1)
    df_grouped = df.groupby(["modality", "algorithm", "n_clusters"], as_index=False).mean(numeric_only=True)
    df_grouped.drop(columns=["run_n", "n_samples"], inplace=True)
    preds_dataset = df[["modality", "algorithm", "n_clusters", "run_n", "sorted_y_pred", "sorted_y_pred_idx"]]
    for alg in preds_dataset["algorithm"].unique():
        df_alg = preds_dataset[preds_dataset["algorithm"] == alg]
        for view in df_alg["modality"].unique():
            df_alg_view = df_alg[df_alg["modality"] == view]
            for cluster in df_alg_view["n_clusters"].unique():
                df_alg_view_clus = df_alg_view[df_alg_view['n_clusters'] == cluster]
                amis= []
                for run_1, run_2 in set(itertools.combinations(df_alg_view_clus["run_n"].unique(), 2)):
                    pred1_alg = df_alg_view_clus.loc[(df_alg_view_clus["run_n"] == run_1), "sorted_y_pred"].to_list()[0]
                    pred2_alg = df_alg_view_clus.loc[(df_alg_view_clus["run_n"] == run_2), "sorted_y_pred"].to_list()[0]
                    pred1_idx = df_alg_view_clus.loc[(df_alg_view_clus["run_n"] == run_1), "sorted_y_pred_idx"].to_list()[0]
                    pred2_idx = df_alg_view_clus.loc[(df_alg_view_clus["run_n"] == run_2), "sorted_y_pred_idx"].to_list()[0]
                    # Only select samples in common for stability metrics
                    common_samples = list(set(pred1_idx) & set(pred2_idx))
                    pred1_common = [pred1_alg[pred1_idx.index(i)] for i in common_samples]
                    pred2_common = [pred2_alg[pred2_idx.index(i)] for i in common_samples]
                    amis.append(adjusted_mutual_info_score(pred1_common, pred2_common))
                df_grouped.loc[(df_grouped["algorithm"] == alg) & (df_grouped["modality"] == view) & (df_grouped["n_clusters"] == cluster), ["AMI"]] = [np.mean(amis)]
    return df_grouped

Get data from previous benchmarkings to compare

In [ ]:
bench1_2clusters = pd.read_csv('../results/cluster_analysis/benchmarking_files/firstbench_2clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_3clusters = pd.read_csv('../results/cluster_analysis/benchmarking_files/firstbench_3clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_4clusters = pd.read_csv('../results/cluster_analysis/benchmarking_files/firstbench_4clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_5clusters = pd.read_csv('../results/cluster_analysis/benchmarking_files/firstbench_5clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
frames = [bench1_2clusters, bench1_3clusters, bench1_4clusters, bench1_5clusters]
results1_file = pd.concat(frames)

In [ ]:
multi_df = results1_file[["view_combination", "algorithm", "n_clusters", "run_n", "n_samples", "y_pred", "y_pred_idx", "silhouette"]]
multi_df.rename(columns={"view_combination":"modality"}, inplace=True)

Calculate stability metrics

In [ ]:
# Function to normalise metrics (adapted)
def add_normalised_metric(df, variable_to_normalise, metric, greater_is_better=True):
    possible_variables = ["algorithm", "modality", "n_clusters"]
    valid_variables = [var for var in possible_variables if var != variable_to_normalise]
    def recursive_loop(subset, remaining_vars, current_filters):
        if not remaining_vars:
            scores = subset[metric].values
            relative_score = scores / scores.max()
            if not greater_is_better:
                relative_score = 1 - relative_score
            condition = True
            for key, value in current_filters.items():
                condition &= (df[key] == value)
            df.loc[condition, f'normalised_{metric}'] = relative_score
            return
        current_var = remaining_vars[0]
        for unique_value in subset[current_var].unique():
            filtered_subset = subset[subset[current_var] == unique_value]
            recursive_loop(filtered_subset, remaining_vars[1:], {**current_filters, current_var: unique_value})
    df[f'normalised_{metric}'] = float('nan')
    recursive_loop(df, valid_variables, {})
    return df

In [ ]:
grouped_sil = pd.concat([singleomics_df, multi_df])
grouped_sil["silhouette"] = grouped_sil["silhouette"].clip(lower=0)
normalised_silhouette = add_normalised_metric(grouped_sil, variable_to_normalise='modality', metric='silhouette', greater_is_better=True)
normalised_silhouette.sort_values(by='normalised_silhouette', ascending=False)

In [ ]:
grouped_df = calc_ami(normalised_silhouette)

In [ ]:
# grouped_df = pd.concat([single_ami, multi_ami])
grouped_df["AMI"] = grouped_df["AMI"].clip(lower=0)
normalised_ami = add_normalised_metric(grouped_df, variable_to_normalise='modality', metric='AMI', greater_is_better=True)
normalised_ami

In [ ]:
grouped_df['combined_metric'] = grouped_df[['normalised_silhouette', 'normalised_AMI']].apply(lambda x: hmean(x), axis=1)
grouped_df.sort_values(["combined_metric"], ascending=False, inplace=True)
grouped_df.replace({"modality": {"CNA": "100000", "Methylation": "010000", "Mutation": "001000", "RNAseq": "000100", "RPPA": "000010", "miRNA": "000001"}}, inplace=True)
grouped_df['n_views'] = grouped_df["modality"].str.count('1')

First, check position across all modalities where single-omics lie

In [ ]:
metric = 'combined_metric'
views = ['CNA', 'Methyl', 'Mutations', 'RNAseq', 'RPPA', 'miRNA']
results1 = grouped_df.copy()

combinations_average = results1.groupby(['modality']).mean(numeric_only=True).sort_values(by=metric, ascending=False)
combinations_sorted = combinations_average.index.tolist()

fig, ax = plt.subplots(4, 1, sharex=True, figsize=(20,7), height_ratios=[0.5, 0.3, 0.1, 0.1])

# First plot: boxplots with combined metric score for each combination
sns.boxplot(data=results1, x='modality', y=metric, ax=ax[0], 
            order=combinations_sorted, width=0.7, color='white', showmeans=True, 
            meanprops={"marker": "^", "markerfacecolor": "green", "markeredgecolor": "green"})
mean_legend = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=8, label='Mean')
# ax[0].legend(handles=[mean_legend], loc='best')
ax[0].set_ylabel('General performance score')
ax[0].set_xlabel('')
ax[0].set_axisbelow(True)
ax[0].set_ylim(-0.05, 1.05)
for line in ax[0].lines:
    line.set_color('black')
    line.set_xdata(line.get_xdata() + 0.5)
for patch in ax[0].patches:
    patch.set_edgecolor('black')
    vertices = patch.get_path().vertices
    vertices[:, 0] += 0.5

# Second plot: heatmap showing modalities present 
views_matrix = combinations_average.reset_index()
views_matrix_expanded = views_matrix['modality'].apply(lambda x: pd.Series(list(x))).astype(int)
views_matrix_expanded.columns = views
views_matrix_expanded.index = views_matrix['modality']
views_ordered = ['Methyl', 'miRNA', 'CNA', 'RNAseq', 'Mutations', 'RPPA'] 
# (^this is a bit of cheating, it is the order from highest to lowest of the modalities as calculated in the next step)
views_matrix_expanded = views_matrix_expanded.reindex(columns=views_ordered)
sns.heatmap(views_matrix_expanded.T, cmap='Blues', linewidths=0.1, linecolor='black', 
            cbar=False, ax=ax[1]).set(xlabel=None)
# ax[1].set_ylabel('Modalities')
ax[1].tick_params(axis='x', bottom=True, labelbottom=False)

# Third plot: heatmap showing number of modalities
unique_nviews_data = combinations_average['n_views'].to_frame()
sns.heatmap(unique_nviews_data.T, cmap='Reds', linewidths=0.1, linecolor='black', square=True,
            cbar=True, cbar_kws=dict(use_gridspec=True, location="bottom", pad=0.2), ax=ax[2], 
            yticklabels='', xticklabels=unique_nviews_data.index).set(xlabel=None)
ax[2].set_ylabel('Number of \nmodalities', labelpad=30, rotation=0, va='center')

# Fourth plot: heatmap showing number of features
features = [2185, 385, 52, 1419, 71, 192]   # From previous step, in same order
feature_sums = {}
for combination in combinations_sorted:
    total = sum(features[i] for i, bit in enumerate(combination) if bit == '1')
    feature_sums[combination] = total
features_df = pd.DataFrame(feature_sums, index=['number of features'])
sns.heatmap(features_df, cmap='Oranges', linewidths=0.1, linecolor='black', cbar=True, 
            cbar_kws=dict(use_gridspec=True, location="bottom", ticks=[123, 2150, 4304], pad=0.2), ax=ax[3], square=True,
            yticklabels='', xticklabels=features_df.columns).set(xlabel=None)
ax[3].set_ylabel('Number of \nfeatures', labelpad=30, rotation=0, va='center')
ax[3].set_xticklabels('')

# plt.tight_layout()
plt.savefig('figures/combinations.svg', bbox_inches='tight')
plt.show()

In [ ]:
def raincloud_plots_variables(df, column_name, metric_name, ax, ylabel):
    mean_values = df.groupby(column_name)[metric_name].mean().sort_values(ascending=False)
    sorted_categories = mean_values.index.tolist()
    metric_subsets = [df[df[column_name] == cat][metric_name].values for cat in sorted_categories]
    pt.RainCloud(x=column_name, y=metric_name, data=df, bw=0.2, palette=[sns.color_palette('colorblind')[0]],
                 width_viol=0.4, ax=ax, orient="v", move=0.2, order=sorted_categories, alpha=0.8)
    means = df.groupby(column_name)[metric_name].mean().loc[sorted_categories]
    sns.scatterplot(x=range(len(sorted_categories)), y=means.values, ax=ax, color=sns.color_palette('colorblind')[2], s=50, marker='^', zorder=10)
    ax.set_xticks(range(len(sorted_categories)))
    ax.set_xticklabels(sorted_categories)
    ax.set_ylabel(ylabel)
    ax.set_ylim(-0.05, 1.05)
    ax.set_axisbelow(True)
    xticks = plt.xticks()
    tick_labels = [text.get_text() for text in xticks[1]]
    pvalue = kruskal(*metric_subsets).pvalue
    if pvalue >= 0.001:
        pvalue_text = f"Kruskal-Wallis, p = {pvalue:.3f}"
    else:
        pvalue_text = f"Kruskal-Wallis, p = {pvalue:.2e}"
    return pvalue, pvalue_text, tick_labels

Plots for algorithm performance, normalised wrt algorithm

In [ ]:
normalised_silhouette1_algs = add_normalised_metric(grouped_sil, variable_to_normalise='algorithm', metric='silhouette', greater_is_better=True)
stability_metrics_results1_algs = calc_ami(normalised_silhouette1_algs)

stability_metrics_results1_algs.replace({"modality": {"CNA": "100000", "Methylation": "010000", "Mutation": "001000", "RNAseq": "000100", "RPPA": "000010", "miRNA": "000001"}}, inplace=True)
stability_metrics_results1_algs['n_views'] = stability_metrics_results1_algs["modality"].str.count('1')
stability_metrics_results1_algs['AMI'] = stability_metrics_results1_algs['AMI'].fillna(value=0)
stability_metrics_results1_algs['AMI'] = stability_metrics_results1_algs['AMI'].clip(lower=0)
normalised_ami1_algs = add_normalised_metric(stability_metrics_results1_algs, variable_to_normalise='algorithm', metric='AMI', greater_is_better=True)
results1_algs = normalised_ami1_algs.copy()
results1_algs['n_views'] = results1_algs['modality'].str.count('1')
results1_algs["normalised_AMI"] = results1_algs["normalised_AMI"].fillna(value=0)
results1_algs['combined_metric'] = results1_algs[['normalised_silhouette', 'normalised_AMI']].mean(axis=1)
results1_algs.sort_values('combined_metric', ascending=False, inplace=True)
results1_algs

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
pvalue, pvalue_text, algs = raincloud_plots_variables(grouped_df, "algorithm", "combined_metric", ax, "General performance score")
ax.set_xlabel("Algorithm")
# ax.legend(title=f"{pvalue_text}", loc=2, bbox_to_anchor=(0, 0.2))
plt.savefig('figures/algs.svg', bbox_inches='tight')

Plots for number of views

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
pvalue, pvalue_text, views = raincloud_plots_variables(grouped_df, "n_views", "combined_metric", ax, "General performance score")
ax.set_xlabel("No. views")
# ax.legend(title=f"{pvalue_text}", loc=2, bbox_to_anchor=(0, 1))
plt.savefig('figures/views.svg', bbox_inches='tight')

Cluster preferences per algorithm, with GPS normalised wrt algorithm

In [ ]:
normalised_silhouette1_clusters = add_normalised_metric(grouped_sil, variable_to_normalise='n_clusters', metric='silhouette', greater_is_better=True)
stability_metrics_results1_clusters = calc_ami(normalised_silhouette1_clusters)

In [ ]:
stability_metrics_results1_clusters.replace({"modality": {"CNA": "100000", "Methylation": "010000", "Mutation": "001000", "RNAseq": "000100", "RPPA": "000010", "miRNA": "000001"}}, inplace=True)
stability_metrics_results1_clusters['n_views'] = stability_metrics_results1_clusters["modality"].str.count('1')
stability_metrics_results1_clusters['AMI'] = stability_metrics_results1_clusters['AMI'].fillna(value=0)
stability_metrics_results1_clusters['AMI'] = stability_metrics_results1_clusters['AMI'].clip(lower=0)
normalised_ami1_clusters = add_normalised_metric(stability_metrics_results1_clusters, variable_to_normalise='n_clusters', metric='AMI', greater_is_better=True)
results1_clusters = normalised_ami1_clusters.copy()
results1_clusters['n_views'] = results1_clusters['modality'].str.count('1')
results1_clusters["normalised_AMI"] = results1_clusters["normalised_AMI"].fillna(value=0)
results1_clusters['combined_metric'] = results1_clusters[['normalised_silhouette', 'normalised_AMI']].mean(axis=1)
results1_clusters.sort_values('combined_metric', ascending=False, inplace=True)
results1_clusters

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
mydict = {}
for alg in results1_clusters["algorithm"].unique():
    pvalue, pvalue_text, cluster_orders = raincloud_plots_variables(results1_clusters[results1_clusters["algorithm"]==alg], "n_clusters", "combined_metric", ax, "General performance score")
    # print(f"{alg}: {plt.xticks}")
    mydict.update({alg:cluster_orders})
plt.close()
mydict